# ssmd market data — DuckDB quickstart

Explore ssmd data locally with **DuckDB**. Two data paths:

1. **Live JSON** — rolling 1-minute OHLCV bars + the symbols that have them
   (`/v1/data/ohlcv/1m`, `/v1/data/ohlcv/1m/symbols`).
2. **Archived Parquet** — daily OHLCV on GCS via short-lived **signed URLs**
   (`/v1/data/download`). DuckDB reads those URLs directly over `httpfs`.

Prereqs: `uv pip install -r requirements.txt`, then `cp .env.example .env` and paste your
`SSMD_API_KEY`. Launch with `uv run --env-file .env jupyter lab` so the key is in the kernel env.

The helpers below validate every API response and fail loudly with a clear
message rather than letting a missing field surface as a cryptic error later.

In [1]:
import os
import requests
import duckdb
import pandas as pd

# Key + base come from the environment. Launch via:
#   uv run --env-file .env jupyter lab
# which loads notebooks/.env into the kernel's environment (no python-dotenv).
API_KEY = os.environ.get("SSMD_API_KEY")
BASE = os.environ.get("SSMD_API_BASE", "https://api.varshtat.com")
assert API_KEY and not API_KEY.endswith("replace_me"), (
    "SSMD_API_KEY is not set. Copy .env.example to .env, paste your key, and start "
    "Jupyter with: uv run --env-file .env jupyter lab"
)
assert BASE.startswith("https://"), f"SSMD_API_BASE must be https, got: {BASE!r}"


def api_get(path, params=None):
    """GET a data-API endpoint, returning a parsed JSON object.

    Raises on HTTP error and on a non-object body, so callers never operate on
    a half-valid response.
    """
    r = requests.get(
        f"{BASE}{path}",
        headers={"X-API-Key": API_KEY},
        params=params or {},
        timeout=30,
    )
    r.raise_for_status()
    body = r.json()
    if not isinstance(body, dict):
        raise ValueError(f"{path}: expected a JSON object, got {type(body).__name__}")
    return body


def require(obj, key, path):
    """Return obj[key], or raise a clear error naming the endpoint."""
    if key not in obj:
        raise KeyError(f"{path}: response missing '{key}' (got keys: {sorted(obj)})")
    return obj[key]


# One in-memory DuckDB connection reused across cells.
con = duckdb.connect()
print("duckdb", duckdb.__version__, "| base", BASE)

duckdb 1.5.4 | base https://api.varshtat.com


## 0. Catalog — what's archived

`/v1/data/feeds` lists the raw archived feeds with their message types, date
coverage, and row counts — a quick way to see what's available (and how far back)
before you query.

In [2]:
feeds = require(api_get("/v1/data/feeds"), "feeds", "/v1/data/feeds")
pd.DataFrame(feeds)[
    ["name", "stream", "messageTypes", "dateMin", "dateMax", "totalFiles", "totalRows"]
]

,name,stream,messageTypes,dateMin,dateMax,totalFiles,totalRows
0,kalshi,crypto,"[ticker, trade]",2026-02-17,2026-06-27,2975,121525719
1,kraken-futures,futures,"[ticker, trade]",2026-02-17,2026-06-03,1879,15426117
2,kraken-spot,spot,"[ticker, trade]",2026-03-08,2026-06-27,1060,1332965


## 1. Discover live `kraken-spot` symbols

`/v1/data/ohlcv/1m/symbols` lists every symbol that currently has a cached 1m
ring for a feed. kraken-spot symbols are pairs like `BTC/USDT`.

In [3]:
p = "/v1/data/ohlcv/1m/symbols"
syms = api_get(p, {"feed": "kraken-spot"})
symbols = require(syms, "symbols", p)
if not symbols:
    raise RuntimeError("No kraken-spot symbols have bars right now — is the feed live?")
print(f"{require(syms, 'count', p)} kraken-spot symbols with 1m bars")
print(symbols)

24 kraken-spot symbols with 1m bars
['ADA/USDT', 'ALEO/USDT', 'ALGO/USDT', 'ATOM/USDT', 'AVAX/USDT', 'BCH/USDT', 'BNB/USDT', 'BTC/USDT', 'DOGE/USDT', 'ETH/USDT', 'FARTCOIN/USDT', 'KAS/USDT', 'LTC/USDT', 'MELANIA/USDT', 'PENGU/USDT', 'SHIB/USDT', 'SOL/USDT', 'TON/USDT', 'TRUMP/USDT', 'VIRTUAL/USDT', 'XAUT/USDT', 'XMR/USDT', 'XRP/USDT', 'XTZ/USDT']


## 2. Live 1m bars → DuckDB

Pull the rolling ~60-bar ring for one symbol, register the rows as a DuckDB
relation, and run SQL over them (VWAP, range, period return).

The ring holds **one bar per minute that had at least one trade**, so for a thin
symbol the 60 bars can span well over an hour with gaps. We use `BTC/USDT`
(liquid), whose minutes are near-contiguous. `start_ts_ms` is each minute's UTC
epoch-ms boundary.

In [4]:
sym = "BTC/USDT"
p = "/v1/data/ohlcv/1m"
bars = require(api_get(p, {"feed": "kraken-spot", "sym": sym, "limit": 60}), "bars", p)
if not bars:
    raise RuntimeError(f"No cached bars for {sym} yet — try another symbol from the list.")

bars_df = pd.DataFrame(bars)
expected_cols = {"o", "h", "l", "c", "v", "start_ts_ms"}
missing = expected_cols - set(bars_df.columns)
assert not missing, f"bar rows missing columns: {missing}"
con.register("bars_df", bars_df)

# start_ts_ms is each minute's UTC epoch-ms boundary; AT TIME ZONE 'UTC' renders
# the UTC wall clock (otherwise to_timestamp shows your local timezone).
summary = con.sql("""
    SELECT
        count(*)                                                  AS n_bars,
        to_timestamp(min(start_ts_ms) / 1000) AT TIME ZONE 'UTC' AS first_bar_utc,
        to_timestamp(max(start_ts_ms) / 1000) AT TIME ZONE 'UTC' AS last_bar_utc,
        min(l)                                                    AS low,
        max(h)                                                    AS high,
        sum(c * v) / nullif(sum(v), 0)                            AS vwap_close,
        arg_min(o, start_ts_ms)                                   AS first_open,
        arg_max(c, start_ts_ms)                                   AS last_close,
        (arg_max(c, start_ts_ms) / nullif(arg_min(o, start_ts_ms), 0) - 1) * 100 AS pct_change
    FROM bars_df
""").df()
print(sym)
summary

BTC/USDT


,n_bars,first_bar_utc,last_bar_utc,low,high,vwap_close,first_open,last_close,pct_change
0,60,2026-06-27 12:52:00,2026-06-27 14:27:00,60367.1,60722.1,60476.923226,60367.1,60627.4,0.431195


In [5]:
# 10 most recent cached bars, oldest → newest.
# NOTE: the ring stores one bar per minute that HAD a trade, so these are the
# last 10 *bars* — not 10 consecutive minutes (quiet minutes are simply absent).
con.sql("""
    SELECT
        to_timestamp(start_ts_ms / 1000) AT TIME ZONE 'UTC' AS bar_minute_utc,
        o, h, l, c, v
    FROM bars_df
    ORDER BY start_ts_ms DESC
    LIMIT 10
""").df().iloc[::-1].reset_index(drop=True)

,bar_minute_utc,o,h,l,c,v
0,2026-06-27 14:11:00,60584.0,60584.0,60584.0,60584.0,0.004184
1,2026-06-27 14:13:00,60639.0,60660.0,60639.0,60660.0,0.005373
2,2026-06-27 14:14:00,60705.9,60722.1,60677.7,60677.7,0.005937
3,2026-06-27 14:15:00,60626.6,60626.6,60605.4,60605.4,0.049306
4,2026-06-27 14:16:00,60632.7,60632.7,60632.7,60632.7,0.004396
5,2026-06-27 14:19:00,60637.1,60637.1,60637.1,60637.1,0.003072
6,2026-06-27 14:20:00,60610.8,60610.8,60610.8,60610.8,0.000181
7,2026-06-27 14:22:00,60619.1,60619.1,60619.1,60619.1,0.000817
8,2026-06-27 14:25:00,60633.3,60633.3,60633.3,60633.3,0.001584
9,2026-06-27 14:27:00,60626.9,60627.4,60626.9,60627.4,0.000847


In [6]:
# Want a true *contiguous* minute grid instead? Build one and gap-fill quiet
# minutes (forward-fill close, volume 0). This is the honest "last 15 minutes".
con.sql("""
    WITH bounds AS (SELECT max(start_ts_ms) AS hi FROM bars_df),
    grid AS (SELECT unnest(range(hi - 14*60000, hi + 60000, 60000)) AS ts FROM bounds),
    joined AS (
        SELECT g.ts, b.c, b.v
        FROM grid g LEFT JOIN bars_df b ON b.start_ts_ms = g.ts
    )
    SELECT
        to_timestamp(ts / 1000) AT TIME ZONE 'UTC'        AS minute_utc,
        last_value(c IGNORE NULLS) OVER (ORDER BY ts)     AS close_ffill,
        coalesce(v, 0)                                    AS v
    FROM joined
    ORDER BY ts
""").df()

,minute_utc,close_ffill,v
0,2026-06-27 14:13:00,60660.0,0.005373
1,2026-06-27 14:14:00,60677.7,0.005937
2,2026-06-27 14:15:00,60605.4,0.049306
3,2026-06-27 14:16:00,60632.7,0.004396
4,2026-06-27 14:17:00,60632.7,0.000000
5,2026-06-27 14:18:00,60632.7,0.000000
6,2026-06-27 14:19:00,60637.1,0.003072
7,2026-06-27 14:20:00,60610.8,0.000181
8,2026-06-27 14:21:00,60610.8,0.000000
9,2026-06-27 14:22:00,60619.1,0.000817


## 3. DuckDB over `httpfs` — read a Parquet straight from a signed URL

`/v1/data/download` returns signed GCS URLs for archived Parquet. The `hols`
feed publishes daily crypto OHLCV (`ohlcv-1m-binance`, `ohlcv-5m-binance`,
`ohlcv-5m-kraken`). DuckDB's `httpfs` extension reads the signed URL directly —
no download step.

Each row carries explicit **provenance**: `source`, `exchange` (`binance`/`kraken`),
`method` (`rest` = polled OHLC/klines, `ws` = aggregated WebSocket trades), and
`interval` (`1m`/`5m`). Note: these columns were added 2026-06-27, so files from
before then won't have them — use `union_by_name := true` when reading a mix.

> hols daily files land ~01:00 UTC for the **prior** day, so we look a day or
> two back.

In [7]:
from datetime import date, timedelta

to_day = (date.today() - timedelta(days=1)).isoformat()
from_day = (date.today() - timedelta(days=3)).isoformat()
p = "/v1/data/download"
files = require(api_get(p, {"feed": "hols", "from": from_day, "to": to_day}), "files", p)


def pick(files, *preferred_types):
    for t in preferred_types:
        for f in files:
            if f.get("type") == t:
                return f
    return files[-1] if files else None


target = pick(files, "ohlcv-1m-binance", "ohlcv-5m-binance", "ohlcv-5m-kraken")
if target is None:
    raise RuntimeError(f"No hols files for {from_day}..{to_day} — widen the date range.")
url = require(target, "signedUrl", p)
print("using:", target.get("path"), f"({target.get('bytes', 0) / 1e6:.1f} MB)")

using: hols/crypto/daily/2026-06-24/ohlcv-1m-binance.parquet (16.7 MB)


In [8]:
# httpfs lets read_parquet() take an https URL. Install/load once per session.
con.execute("INSTALL httpfs; LOAD httpfs;")

# Signed URLs contain no single quotes, so inlining is safe.
con.execute(f"DESCRIBE SELECT * FROM read_parquet('{url}')").df()

,column_name,column_type,null,key,default,extra
0,symbol,VARCHAR,YES,None,None,None
1,hols_ticker,VARCHAR,YES,None,None,None
2,source,VARCHAR,YES,None,None,None
3,date,TIMESTAMP,YES,None,None,None
4,date_close,TIMESTAMP,YES,None,None,None
5,unix,BIGINT,YES,None,None,None
6,close_unix,BIGINT,YES,None,None,None
7,open,DOUBLE,YES,None,None,None
8,high,DOUBLE,YES,None,None,None
9,low,DOUBLE,YES,None,None,None


In [9]:
# Top 10 symbols by traded volume in this file.
con.execute(f"""
    SELECT
        symbol,
        source,
        count(*)               AS bars,
        sum(volume)            AS total_volume,
        sum(tradecount)        AS trades,
        min(low)               AS day_low,
        max(high)              AS day_high,
        arg_max(close, date)   AS last_close
    FROM read_parquet('{url}')
    GROUP BY symbol, source
    ORDER BY total_volume DESC
    LIMIT 10
""").df()

,symbol,source,bars,total_volume,trades,day_low,day_high,last_close
0,PEPEUSDT,binance_spot,1440,8.843837e+12,69136.0,2.420000e-06,2.720000e-06,2.530000e-06
1,BTTCUSDT,binance_spot,1440,1.548847e+12,8913.0,2.600000e-07,2.800000e-07,2.700000e-07
2,SHIBUSDT,binance_spot,1440,8.444029e+11,34849.0,4.210000e-06,4.600000e-06,4.380000e-06
3,BONKUSDT,binance_spot,1440,5.171500e+11,21428.0,4.010000e-06,4.440000e-06,4.210000e-06
4,FLOKIUSDT,binance_spot,1440,6.169964e+10,109826.0,2.188000e-05,2.420000e-05,2.308000e-05
5,1000SATSUSDT,binance_spot,1440,5.224478e+10,6067.0,8.500000e-06,9.550000e-06,9.060000e-06
6,LUNCUSDT,binance_spot,1440,4.977786e+10,48526.0,5.787000e-05,6.483000e-05,6.077000e-05
7,XECUSDT,binance_spot,1440,3.122036e+10,2929.0,4.850000e-06,5.280000e-06,5.060000e-06
8,NEIROUSDT,binance_spot,1440,2.540440e+10,87300.0,5.758000e-05,6.431000e-05,6.072000e-05
9,DOGSUSDT,binance_spot,1440,2.496056e+10,12494.0,3.650000e-05,4.070000e-05,4.000000e-05


## 4. Tips

**Read many days at once** — `read_parquet` takes a list. Use `union_by_name`
so a mix of older files (no provenance columns) and newer ones still lines up:
```python
urls = [f["signedUrl"] for f in files if f["type"] == "ohlcv-1m-binance"]
con.execute("SELECT count(*) FROM read_parquet(?, union_by_name := true)", [urls]).fetchone()
```

**Provenance** — group/filter by how a bar was sourced (only on files from
2026-06-27 onward):
```python
con.execute(f"SELECT DISTINCT exchange, method, interval, source FROM read_parquet('{url}')").df()
```

**Persist locally** so you don't re-download each session (see cell below).

**Big files** — stream to disk first, then query the local copy:
```python
import requests, os
os.makedirs("data", exist_ok=True)
with requests.get(url, stream=True) as r:
    r.raise_for_status()
    with open("data/hols.parquet", "wb") as fh:
        for chunk in r.iter_content(1 << 20):
            fh.write(chunk)
con.sql("SELECT count(*) FROM 'data/hols.parquet'")
```

**Key scopes/feeds** — this all needs `datasets:read` plus access to each feed
you query (`kraken-spot`, `hols`, …). Manage keys at
<https://harman.varshtat.com> → Admin → API Keys.

In [10]:
# Persist the Parquet into a local DuckDB file for repeat analysis.
import os

os.makedirs("data", exist_ok=True)
pcon = duckdb.connect("data/ssmd.duckdb")
pcon.execute("INSTALL httpfs; LOAD httpfs;")
pcon.execute(f"CREATE OR REPLACE TABLE hols_ohlcv AS SELECT * FROM read_parquet('{url}')")
rows = pcon.execute("SELECT count(*) FROM hols_ohlcv").fetchone()[0]
assert rows > 0, "hols_ohlcv table is empty — the source Parquet had no rows."
print("rows:", rows)
pcon.close()  # reopen later with duckdb.connect('data/ssmd.duckdb')

rows: 629280
